# 📄 논문 Q&A 에이전트 만들기
### LangChain + Pinecone + OpenAI

arXiv 논문을 모아 Pinecone 벡터 DB에 저장하고, 질문에 따라 **스스로 검색 도구를 골라 호출**해
논문 내용을 근거로 답하는 LLM 에이전트를 만듭니다.

**구성 파일**

| 파일 | 역할 |
|---|---|
| `paper_agent.ipynb` (이 노트북) | 원리를 단계별로 확인 |
| `ingest.py` | 논문 수집·적재 (주제별 arXiv 검색 → 청킹 → Pinecone) |
| `papers.json` | 적재된 논문 목록(매니페스트) — `ingest.py`가 갱신 |
| `paper_agent.py` | 검색 도구 + 에이전트 정의 |
| `app.py` | Streamlit 채팅 UI |

**전체 흐름**
1. 환경 설정
2. 데이터 수집 — arXiv에서 논문 PDF 로드 (원리 확인)
3. 전처리 — 청크 분할
4. 적재 — `ingest.py`로 대량 수집
5. 검색 도구(Tool)
6. 에이전트 생성
7. 테스트 & 대화

**RAG(Retrieval-Augmented Generation)란?**
LLM이 학습하지 않은 문서(최신 논문 등)를 외부 DB에 저장해 두고, 질문이 들어오면 관련 부분을 검색해
프롬프트에 넣어주는 기법입니다. 환각(hallucination)을 줄이고 출처를 제시할 수 있다는 장점이 있습니다.

## 1. 환경 설정

API 키는 프로젝트 루트(`SKN/LLM/`)의 `.env` 파일에서 불러옵니다.

```
OPENAI_API_KEY=sk-...
PINECONE_API_KEY=...
```

In [ ]:
%pip install -q -U langchain langchain-openai langchain-pinecone streamlit \
    langchain-text-splitters pinecone python-dotenv arxiv pymupdf requests cohere

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()   # 상위 폴더의 .env 파일을 읽어 환경변수로 등록
from dotenv import load_dotenv
import os
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['TAVILY_API_KEY'] = os.getenv('TAVILY_API_KEY')

os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
for name in ["OPENAI_API_KEY", "PINECONE_API_KEY"]:
    assert os.getenv(name), f".env에 {name}가 없습니다"

INDEX_NAME  = "paper-agent"
CHAT_MODEL  = "gpt-5.6-luna"
EMBED_MODEL = "text-embedding-3-small"   # 1536차원
EMBED_DIM   = 1536
print("설정 완료")

설정 완료


## 2. 데이터 수집 — arXiv 논문 로드

arXiv API로 논문 메타데이터를 조회하고, PDF를 내려받아 PyMuPDF로 텍스트를 추출합니다.
여기서는 **원리를 확인하려고 1편만** 불러옵니다. 실제 대량 수집은 4단계의 `ingest.py`가 담당합니다.

> `langchain_community`의 `ArxivLoader`는 `arxiv` 2.0에서 제거된 `Search.results()`를 호출하기 때문에
> 최신 `arxiv` 패키지(4.x)와 함께 쓰면 `AttributeError`가 납니다. 그래서 직접 받아옵니다.

In [ ]:
import arxiv, requests, pymupdf
from langchain_core.documents import Document

client = arxiv.Client()

def load_arxiv(paper_id: str) -> Document:
    """arXiv에서 논문 PDF를 받아 텍스트 Document로 변환합니다."""
    meta = next(client.results(arxiv.Search(id_list=[paper_id])))    # 메타데이터 조회
    pdf_bytes = requests.get(meta.pdf_url, timeout=60).content       # PDF 다운로드
    with pymupdf.open(stream=pdf_bytes, filetype="pdf") as pdf:      # 페이지별 텍스트 추출
        text = "\n".join(page.get_text() for page in pdf)
    return Document(
        page_content=text,
        metadata={
            "paper_id": paper_id,
            "Title": meta.title,
            "Authors": ", ".join(a.name for a in meta.authors),
            "Published": meta.published.strftime("%Y-%m-%d"),
        },
    )

doc = load_arxiv("1706.03762")   # Attention Is All You Need
print(f"{doc.metadata['Title']} ({len(doc.page_content):,}자)")

In [ ]:
# 로드된 논문 미리보기
print(doc.metadata["Authors"][:120])
print()
print(doc.page_content[:500])

## 3. 전처리 — 청크 분할

논문 한 편은 수만 자라 임베딩 모델의 입력 한도를 넘고, 검색 정밀도도 떨어집니다.
`RecursiveCharacterTextSplitter`로 문단 → 문장 → 단어 순서로 자연스럽게 끊어 약 1000자 단위 청크를 만들고, 문맥이 끊기지 않도록 150자를 겹치게(overlap) 합니다.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_documents([doc])

print(f"논문 1편 → 청크 {len(chunks)}개")
print(chunks[10].page_content[:300])

## 4. 적재 — `ingest.py`로 대량 수집

앞의 2~3단계를 논문 수백 편에 반복하는 것이 `ingest.py`입니다. 노트북에서 직접 돌리지 않고
스크립트로 분리한 이유는 세 가지입니다.

1. **증분 적재** — `papers.json`에 이미 있는 논문은 건너뜁니다. 다시 실행해도 새 논문만 받습니다.
2. **중단 안전성** — 논문 1편을 처리할 때마다 매니페스트를 저장해서, 중간에 끊겨도 진행분이 남습니다.
3. **중복 방지** — 청크 내용으로 만든 결정론적 id를 쓰기 때문에 같은 청크는 덮어쓰기(upsert)됩니다.
   id를 지정하지 않으면 실행할 때마다 새 UUID가 발급돼 같은 내용이 계속 쌓입니다.

`ingest.py`의 `TOPICS`에 주제와 arXiv 검색식을 적어두면, 주제마다 논문을 모아 `topic` 메타데이터와
함께 저장합니다. 이 값으로 나중에 주제별 검색 범위를 좁힐 수 있습니다.

```bash
python ingest.py              # 주제당 10편씩, 아직 없는 논문만 적재
python ingest.py --per 20     # 주제당 20편씩
python ingest.py --reset      # 인덱스를 비우고 처음부터
```

In [ ]:
# 적재 결과 확인 (ingest.py 실행 후)
import json
from collections import Counter
from pathlib import Path

papers = json.loads(Path("papers.json").read_text(encoding="utf-8"))
by_topic = Counter(p["topic"] for p in papers.values())

print(f"적재된 논문: {len(papers)}편\n")
for topic, n in by_topic.most_common():
    print(f"  {n:3d}편  {topic}")

In [ ]:
# Pinecone 인덱스 상태
import os
from dotenv import load_dotenv
from pinecone import Pinecone

load_dotenv()
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
stats = pc.Index("paper-agent").describe_index_stats()

print("벡터 수:", f"{stats['total_vector_count']:,}")
print("차원:", stats["dimension"])
print(f"논문 1편당 평균 {stats['total_vector_count'] / len(papers):.0f}청크")

### 4-1. 검색 테스트

에이전트에 붙이기 전에 벡터 검색 자체를 확인합니다.
`paper_agent.py`가 만들어 둔 것과 같은 방식으로 인덱스에 연결합니다.

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

vectorstore = PineconeVectorStore(
    index_name="paper-agent",
    embedding=OpenAIEmbeddings(model="text-embedding-3-small"),
)

for d in vectorstore.similarity_search("What is multi-head attention?", k=3):
    print(f"[{d.metadata['title'][:45]}]")
    print(f"  {d.page_content[:120]}...\n")

## 5. 검색 도구(Tool) 정의

에이전트는 도구의 **이름과 docstring을 읽고** 언제 쓸지 스스로 판단합니다. 그래서 설명을 명확하게
쓰는 게 중요합니다. 도구 구현은 `paper_agent.py`의 `build_tools()`에 있습니다.

**저장된 논문 (Pinecone) — 본문까지 읽을 수 있음**

| 도구 | 역할 |
|---|---|
| `search_papers` | 저장된 논문에서 검색 (`topic`으로 주제 한정 가능) |
| `search_in_paper` | `paper_id`로 **특정 논문 안에서만** 검색 |
| `find_paper` | 제목·주제 키워드로 논문의 arXiv 번호 찾기 |
| `list_papers` | 주제별 편수 요약 |

**저장되지 않은 논문 — 제목·초록만 확인 가능**

| 도구 | 신호 | 출처 |
|---|---|---|
| `trending_papers` | **추천 수**(화제성) | Hugging Face Daily Papers |
| `latest_papers` | **등록일**(분야별 최신) | arXiv `cat:` 분류 |
| `search_new_papers` | **주제 관련도** | arXiv 초록 검색 |

"요즘 핫한 논문"과 "최신 논문"은 다른 질문입니다. arXiv는 인용수·조회수 같은 인기 지표를 제공하지
않아 **최신순**만 알 수 있고, 화제성은 커뮤니티가 추천을 누르는 Hugging Face Daily Papers에서
가져옵니다. 대신 이쪽은 AI·머신러닝에 치우쳐 있어서, 금융처럼 다른 분야의 동향은 `latest_papers`로 봐야 합니다.

### 논문이 많아지면 벡터 검색만으로는 부족합니다

논문이 수백 편이 되면 질문과 무관한 분야의 청크가 상위에 올라옵니다. 그래서 2단계로 검색합니다.

```
질문 → 벡터 검색 20개(CANDIDATE_K) → Cohere 리랭커 → 상위 4개(FINAL_K) → LLM
```

벡터 검색은 질문과 청크를 **각각 따로** 벡터로 만들어 비교하지만, 리랭커는 질문과 청크를
**함께 읽고** 점수를 매기기 때문에 훨씬 정확합니다. 대신 느리고 비용이 들어서, 먼저 벡터 검색으로
후보를 줄인 뒤 리랭커를 적용합니다. `COHERE_API_KEY`가 없으면 벡터 검색 결과를 그대로 씁니다.

In [ ]:
from paper_agent import build_tools, load_papers

papers = load_papers()
tools = build_tools(vectorstore, papers)

for t in tools:
    print(f"● {t.name}{tuple(t.args.keys())}")
    print(f"   {t.description.splitlines()[0]}")

## 6. 에이전트 생성

`create_agent`는 LLM이 **[생각 → 도구 호출 → 결과 확인 → 답변]** 루프를 돌 수 있게 해줍니다.
`InMemorySaver`를 checkpointer로 넣으면 `thread_id` 별로 대화 이력이 저장돼
후속 질문("그건 왜 그래?")도 이해합니다.

`paper_agent.py`의 `build_agent()`가 인덱스 연결 → 도구 생성 → 에이전트 조립까지 한 번에 처리합니다.
`app.py`도 같은 함수를 쓰기 때문에, 노트북에서 확인한 동작이 웹 UI에서도 그대로 재현됩니다.

In [ ]:
from paper_agent import build_agent, SYSTEM_PROMPT

agent, papers = build_agent()

print("--- 시스템 프롬프트 ---")
print(SYSTEM_PROMPT[:400], "...")
print(f"\n논문 {len(papers)}편으로 에이전트 준비 완료")

In [ ]:
import uuid
from langchain_core.messages import HumanMessage

def ask(question: str, thread_id: str, show_tools: bool = True) -> str:
    """한 턴 대화. 같은 thread_id를 쓰면 이전 대화를 기억합니다."""
    result = agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config={"configurable": {"thread_id": thread_id}},
    )
    messages = result["messages"]

    if show_tools:
        # checkpointer 때문에 이전 턴 메시지도 함께 들어있으므로, 마지막 질문 이후만 확인
        turn_start = max(i for i, m in enumerate(messages) if isinstance(m, HumanMessage))
        for m in messages[turn_start:]:
            for tc in getattr(m, "tool_calls", None) or []:
                print(f"  🔧 도구 호출: {tc['name']}({tc['args']})")

    # 추론 모델의 응답은 content가 블록 리스트(추론+답변)라 .text로 본문만 꺼냅니다
    return messages[-1].text

thread = str(uuid.uuid4())

## 7. 테스트

In [12]:
print(ask("어떤 논문들이 있어?", thread))

  🔧 도구 호출: list_papers({})
현재 읽을 수 있는 논문은 다음 3편입니다.

1. **Attention Is All You Need** (arXiv:1706.03762)
2. **BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding** (arXiv:1810.04805)
3. **Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks** (arXiv:2005.11401)

📎 출처: 위 3편의 저장된 논문 목록


C:\Users\UK\AppData\Local\Temp\ipykernel_77836\4250220790.py:1: LangChainDeprecationWarning: Calling .text() as a method is deprecated. Use .text as a property instead (e.g., message.text).
  print(ask("어떤 논문들이 있어?", thread))


In [13]:
print(ask("Transformer에서 self-attention이 RNN보다 좋은 이유가 뭐야?", thread))

  🔧 도구 호출: search_papers({'query': 'Why is self-attention better than recurrent neural networks RNN Transformer advantages parallelization path length computational complexity'})
  🔧 도구 호출: search_in_paper({'query': 'Table 1 self-attention recurrent layer complexity sequential operations maximum path length O(n^2 d) O(n d^2)', 'paper_id': '1706.03762'})
  🔧 도구 호출: search_in_paper({'query': 'self-attention layers are faster than recurrent layers when sequence length n is smaller than representation dimension d example translation sentence lengths', 'paper_id': '1706.03762'})
Transformer의 **self-attention이 RNN보다 유리한 이유**는 주로 세 가지입니다.

1. **병렬 계산이 가능함**
   - RNN은 이전 시점의 은닉 상태가 계산되어야 다음 시점을 계산할 수 있어, 토큰을 순차적으로 처리해야 합니다.
   - 반면 self-attention은 한 층에서 모든 토큰 간의 관계를 동시에 계산할 수 있습니다.
   - 따라서 GPU·TPU에서 더 효율적으로 학습할 수 있습니다.
   - 논문에서는 필요한 순차 연산 수를 RNN은 **O(n)**, self-attention은 **O(1)**로 비교합니다.

2. **먼 거리의 토큰도 직접 연결함**
   - RNN에서 문장 앞부분과 뒷부분의 정보가 연결되려면 여러 시점을 거쳐야 합니다. 문장이 길수록 정보 전달 경로도 길어집니다.
   -

C:\Users\UK\AppData\Local\Temp\ipykernel_77836\3283059312.py:1: LangChainDeprecationWarning: Calling .text() as a method is deprecated. Use .text as a property instead (e.g., message.text).
  print(ask("Transformer에서 self-attention이 RNN보다 좋은 이유가 뭐야?", thread))


In [14]:
# 후속 질문 — 앞 대화의 맥락(Transformer)을 기억하는지 확인
print(ask("그럼 그 구조에서 positional encoding은 왜 필요해?", thread))

  🔧 도구 호출: search_in_paper({'query': 'Why positional encoding is needed because Transformer contains no recurrence or convolution sequence order sinusoidal positional encodings', 'paper_id': '1706.03762'})
  🔧 도구 호출: search_in_paper({'query': 'Since our model contains no recurrence and no convolution order of sequence positional encodings input embeddings', 'paper_id': '1706.03762'})
Positional encoding은 **Transformer가 각 토큰의 순서와 위치를 알 수 있게 해주는 정보**입니다.

### 왜 필요한가?

RNN은 토큰을 순서대로 처리합니다.

> 첫 번째 토큰 → 두 번째 토큰 → 세 번째 토큰

따라서 계산 과정 자체에 순서 정보가 들어 있습니다. 반면 Transformer의 self-attention은 모든 토큰을 한꺼번에 처리합니다. 이때 위치 정보가 없다면 다음 두 문장을 구분하기 어려워집니다.

- `고양이가 개를 쫓았다`
- `개가 고양이를 쫓았다`

두 문장은 같은 단어를 포함하지만, 단어의 **순서**가 다릅니다. Positional encoding은 각 토큰의 표현에 위치 정보를 더해 이런 순서를 구분하도록 합니다.

### 어떻게 사용하는가?

논문에서는 입력 임베딩에 positional encoding을 **더해서** 사용합니다.

\[
\text{Input representation}
=
\text{Token embedding}
+
\text{Positional encoding}
\]

Positional encoding은 토큰 임베딩과 같은 차원을 가지므로 두 벡터를 더할 수 있습니다.

### Transfor

C:\Users\UK\AppData\Local\Temp\ipykernel_77836\1628283144.py:2: LangChainDeprecationWarning: Calling .text() as a method is deprecated. Use .text as a property instead (e.g., message.text).
  print(ask("그럼 그 구조에서 positional encoding은 왜 필요해?", thread))


In [15]:
print(ask("BERT와 RAG 논문의 접근 방식을 비교해줘", thread))

  🔧 도구 호출: search_in_paper({'query': 'BERT approach bidirectional Transformer pre-training masked language model next sentence prediction fine-tuning downstream tasks', 'paper_id': '1810.04805'})
  🔧 도구 호출: search_in_paper({'query': 'RAG approach retrieval augmented generation retriever document index generator question answering parametric non-parametric memory training', 'paper_id': '2005.11401'})
BERT와 RAG는 모두 Transformer를 활용하지만, **해결하려는 문제와 지식을 사용하는 방식**이 다릅니다.

| 구분 | BERT | RAG |
|---|---|---|
| 핵심 목적 | 문장의 의미를 깊게 이해하는 표현 학습 | 외부 문서를 검색해 지식 기반 답변·텍스트 생성 |
| 기본 구조 | Transformer **encoder** 기반 | **검색기(retriever) + 생성기(generator)** 결합 |
| 지식 저장 방식 | 모델 파라미터에 지식 저장 | 모델 파라미터의 지식 + 외부 문서 저장소의 지식 |
| 주요 학습 방식 | Masked Language Model, Next Sentence Prediction | 입력에 관련된 증거 문서를 검색하고, 이를 바탕으로 생성 |
| 주요 활용 | 분류, 질의응답, 자연어 추론 등 | Open-domain QA, 지식 집약적 생성 |
| 외부 지식 사용 | 기본적으로 입력에 주어진 텍스트만 사용 | 검색된 외부 문서를 함께 사용 |

### 1. BERT: 문맥을 이해하는 모델

BERT는 Transformer encoder를 이용해 문장의 양방향 문맥을 학습합니다.

- 

C:\Users\UK\AppData\Local\Temp\ipykernel_77836\775872794.py:1: LangChainDeprecationWarning: Calling .text() as a method is deprecated. Use .text as a property instead (e.g., message.text).
  print(ask("BERT와 RAG 논문의 접근 방식을 비교해줘", thread))


In [16]:
# 저장된 논문에 없는 내용 → 환각 없이 모른다고 답하는지 확인
print(ask("이 논문들에서 GPT-4의 파라미터 수가 얼마라고 나와?", thread))

  🔧 도구 호출: search_papers({'query': 'GPT-4 number of parameters parameter count'})
저장된 논문들에는 **GPT-4의 파라미터 수가 나오지 않습니다.**

검색 결과:

- **BERT 논문**에는 2018년의 OpenAI GPT가 언급되지만, GPT-4의 파라미터 수는 제시되지 않습니다.
- **RAG 논문**에는 T5-11B의 110억 개 파라미터, T5-large의 7억 7천만 개 파라미터 등이 언급되지만, GPT-4에 대한 파라미터 수는 없습니다.

또한 GPT-4의 정확한 파라미터 수는 공개적으로 공식 확인되지 않은 정보로 알려져 있습니다. 따라서 이 논문들을 근거로 GPT-4의 파라미터 수를 특정할 수 없습니다.

📎 출처: *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*, *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*


C:\Users\UK\AppData\Local\Temp\ipykernel_77836\2496895815.py:2: LangChainDeprecationWarning: Calling .text() as a method is deprecated. Use .text as a property instead (e.g., message.text).
  print(ask("이 논문들에서 GPT-4의 파라미터 수가 얼마라고 나와?", thread))


## 8. 자유 대화
아래 셀을 실행하면 터미널처럼 대화할 수 있습니다. (종료: `quit`)

In [ ]:
chat_thread = str(uuid.uuid4())
while True:
    q = input("You: ").strip()
    if q.lower() in {"quit", "exit", "q"}:
        break
    if q:
        print("Bot:", ask(q, chat_thread, show_tools=False), "\n")

## 9. 정리 & 확장 아이디어

**구현한 것**

| 단계 | 사용 기술 | 역할 |
|---|---|---|
| 데이터 수집 | `arxiv` + `pymupdf` | 주제별 arXiv 검색 → PDF → 텍스트 |
| 전처리 | `RecursiveCharacterTextSplitter` | 1000자 청크, 150자 overlap |
| 임베딩 | OpenAI `text-embedding-3-small` | 텍스트 → 1536차원 벡터 |
| 벡터 DB | Pinecone Serverless | 코사인 유사도 검색 + `topic`/`paper_id` 필터 |
| 재순위 | Cohere `rerank-v3.5` | 후보 20개 → 상위 4개 |
| 에이전트 | LangChain `create_agent` + `InMemorySaver` | 도구 선택·호출, 대화 메모리 |
| 웹 UI | Streamlit (`app.py`) | 채팅 화면, 도구 호출 실시간 표시 |

**Pinecone에 실제로 저장되는 것**

벡터 1건 = `1536차원 임베딩` + `메타데이터`이고, 메타데이터에는 **청크 원문(`text`)** 이 함께 들어갑니다.
그래야 검색 결과를 `page_content`로 돌려줄 수 있습니다. 즉 논문 본문은 청크 단위로 쪼개져
Pinecone에 그대로 보관됩니다. `app.py`가 PDF를 다시 받지 않고도 동작하는 이유입니다.

검색은 **읽기 전용**입니다. 질문·답변·대화 내용은 Pinecone에 저장되지 않습니다.
대화 기억은 `InMemorySaver`가 파이썬 프로세스 메모리에 들고 있는 것이라 재시작하면 사라집니다.

**확장해 볼 것**
- 섹션(Abstract/Method/Experiments) 인식 청킹 → 참고문헌 목록이 검색되는 문제 완화
- 하이브리드 검색(키워드 + 벡터)으로 고유명사·수식 검색 정확도 향상
- Agentic RAG 그래프(검색 결과 평가 → 질문 재작성 루프) 도입
- 에이전트가 직접 논문을 적재하는 도구 + 승인 미들웨어(HITL)
- 답변 품질 평가: RAGAS 등으로 정확도·근거 충실도 측정

**주의**
- 임베딩 모델을 바꾸면 차원이 달라지므로 새 인덱스를 만들어야 합니다.
- 논문을 늘리려면 `ingest.py`의 `TOPICS`에 주제를 추가하고 다시 실행하세요.
  이미 있는 논문은 건너뛰므로 추가된 만큼만 시간이 걸립니다.